<a href="https://colab.research.google.com/github/nathirkasim/gitdemo/blob/main/skillrecommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# --- STEP 1: LOAD AND PREPARE DATA ---

def load_data(filepath):
    """Loads and cleans the job data."""
    try:
        df = pd.read_csv(filepath)
        df['Required_Skills'] = df['Required_Skills'].fillna('')
        df['Job_Title'] = df['Job_Title'].str.strip().str.lower()
        return df
    except Exception as e:
        print("Error loading job data:", e)
        return None

def load_user_data(filepath):
    """Loads and cleans the user data."""
    try:
        df = pd.read_csv(filepath)
        df['target_job_title'] = df['target_job_title'].str.strip().str.lower()
        df['current_skills'] = df['current_skills'].str.lower()
        return df
    except Exception as e:
        print("Error loading user data:", e)
        return None


# --- STEP 2: BUILD THE CORE ML MODEL ---

class SkillRecommender:
    def __init__(self):
        # Word-level TF-IDF, include unigrams and bigrams
        self.vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1, 2))
        self.role_archetypes = {}
        self.job_data = None

    def train(self, jobs_df):
        """Trains the vectorizer and creates an ideal 'archetype' vector for each role."""
        print("Training model...")
        self.job_data = jobs_df

        # Create a TF-IDF matrix for all job skills
        tfidf_matrix = self.vectorizer.fit_transform(self.job_data['Required_Skills'])

        # For each unique job title, calculate its average skill vector (archetype)
        for role in self.job_data['Job_Title'].unique():
            indices = self.job_data[self.job_data['Job_Title'] == role].index
            role_vectors = tfidf_matrix[indices]
            self.role_archetypes[role] = role_vectors.mean(axis=0).A1  # dense 1D array
        print("Model training complete.")

    def predict(self, user_skills_str, target_role):
        """Generates recommendations for a user based on their skills."""
        if not self.role_archetypes:
            raise RuntimeError("Model has not been trained. Please call .train() first.")

        # Convert user's skills into a TF-IDF vector using the trained vectorizer
        user_vector = self.vectorizer.transform([user_skills_str]).toarray()

        # Calculate similarity scores between user's skills and all role archetypes
        scores = {}
        for role, archetype in self.role_archetypes.items():
            score = cosine_similarity(user_vector, archetype.reshape(1, -1))
            scores[role] = score[0][0]

        # Find the role with the highest match score
        best_match_role = max(scores, key=scores.get)

        # --- Generate Recommendation Based on Best Match ---
        if best_match_role != target_role:
            status = "role_pivot"
            message = f"Your skills are a stronger match for the role of: {best_match_role.title()}"
            role_to_get_skills_from = best_match_role
        else:
            status = "good_match"
            message = "You are on the right track! To be even stronger in this role, consider learning:"
            role_to_get_skills_from = target_role

        # Get the top skills for the role we are recommending for
        all_skills_for_role = ' '.join(
            self.job_data[self.job_data['Job_Title'] == role_to_get_skills_from]['Required_Skills']
        )
        all_skills_vector = self.vectorizer.transform([all_skills_for_role])

        # Find features (skills) with the highest TF-IDF scores for that role
        feature_array = np.array(self.vectorizer.get_feature_names_out())
        tfidf_sorting = np.argsort(all_skills_vector.toarray()).flatten()[::-1]
        top_skills = feature_array[tfidf_sorting][:30]  # take more, then filter down

        # --- Filtering logic (Option 2) ---
        user_skills_set = set(skill.strip() for skill in user_skills_str.split(','))
        skill_recs = []
        seen_words = set()

        for skill in top_skills:
            words = skill.split()

            # If it's a bigram (2 words), prefer it
            if len(words) == 2 and skill not in user_skills_set:
                skill_recs.append(skill)
                seen_words.update(words)

            # If it's a unigram, only keep it if it's not part of a recommended bigram
            elif len(words) == 1 and skill not in user_skills_set and words[0] not in seen_words:
                skill_recs.append(skill)

            if len(skill_recs) >= 5:
                break

        return status, message, skill_recs


# --- STEP 3: RUN THE MODEL ---

jobs_df = load_data('final_data.csv')
users_df = load_user_data('sample_user_data.csv')

if jobs_df is not None and users_df is not None:
    model = SkillRecommender()
    model.train(jobs_df)

    print("\n" + "="*50 + "\n")

    for index, user_profile in users_df.iterrows():
        print(f"--- Recommendations for {user_profile['user_id']} ---")
        print(f"Target Role: {user_profile['target_job_title'].title()}")
        print(f"Current Skills: {user_profile['current_skills']}")
        print("-" * 20)

        status, message, skill_recs = model.predict(
            user_profile['current_skills'],
            user_profile['target_job_title']
        )

        print(message)
        if skill_recs:
            for i, skill in enumerate(skill_recs, 1):
                print(f"{i}. {skill.title()}")

        print("\n" + "="*50 + "\n")


Training model...
Model training complete.


--- Recommendations for student_1 ---
Target Role: Business Analyst
Current Skills: sql, tableau, data analysis
--------------------
Your skills are a stronger match for the role of: Business_Analyst
1. Data_Analysis
2. Business_Intelligence
3. Requirements_Analysis
4. Agile
5. Data_Analysis Requirements_Analysis


--- Recommendations for student_2 ---
Target Role: Web Developer
Current Skills: html, css, javascript
--------------------
Your skills are a stronger match for the role of: Web_Developer
1. Js
2. Node Js
3. React
4. Angular
5. Vue


--- Recommendations for student_3 ---
Target Role: Graphics Designer
Current Skills: figma, sketch, ui/ux design, photoshop, illustrator, adobe creative suite
--------------------
Your skills are a stronger match for the role of: Graphics_Designer
1. Ui
2. Ui Ux_Design
3. Invision
4. Adobe_Creative_Suite
5. Illustrator Ui


--- Recommendations for student_4 ---
Target Role: Database Administrator
Curr